# LCO header fetcher (resumable)

Notebook version of `lco_fetch_resume.py`. Run the cells in order: 1) definitions, 2) settings, 3) fetch, 4) export, 5) load into pandas.

Fetching is checkpointed in the SQLite file, so you can stop a cell (Kernel > Interrupt) and run it again to resume.

In [ ]:
import csv
import getpass
import json
import os
import sqlite3
import sys
import tempfile
import time
from contextlib import closing
from datetime import date, datetime, timedelta, timezone
from http.client import IncompleteRead
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urljoin, urlsplit
from urllib.request import Request, urlopen
from types import SimpleNamespace

In [ ]:
ARCHIVE = "https://archive-api.lco.global"
ONE_DAY = timedelta(days=1)
FRAME_FIELDS = ("basename", "observation_date", "site_id", "telescope_id", "instrument_id")
HEADER_FIELDS = (
    "WCSERR", "SATFRAC", "EXPTIME", "FILTER", "AIRMASS", "L1FWHM",
    "MOONFRAC", "MOONDIST", "SKYMAG", "WMSHUMID", "WMSTEMP", "PIXSCALE",
)

In [ ]:
def get_json(path, token):
    url = urljoin(ARCHIVE, path)
    parts = urlsplit(url)
    if (parts.scheme, parts.netloc) != ("https", "archive-api.lco.global"):
        raise ValueError(f"Unexpected archive pagination URL: {url}")
    request = Request(url, headers={"Accept": "application/json"})
    if token:
        request.add_unredirected_header("Authorization", f"Token {token}")

    for attempt in range(4):
        time.sleep(0.1)
        try:
            with urlopen(request, timeout=45) as response:
                return json.load(response)
        except HTTPError as error:
            error.close()
            if error.code not in (500, 502, 503, 504) or attempt == 3:
                raise
        except (URLError, TimeoutError, ConnectionError, IncompleteRead):
            if attempt == 3:
                raise
        time.sleep(2 ** attempt)


def open_checkpoint(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    database = sqlite3.connect(path)
    database.execute("PRAGMA synchronous = FULL")
    database.executescript("""
        CREATE TABLE IF NOT EXISTS listed_days (
            day TEXT PRIMARY KEY
        );
        CREATE TABLE IF NOT EXISTS frames (
            id INTEGER PRIMARY KEY,
            day TEXT NOT NULL,
            metadata TEXT NOT NULL,
            headers TEXT,
            header_error INTEGER
        );
        CREATE INDEX IF NOT EXISTS frames_by_day ON frames(day);
    """)
    return database


def inventory_day(database, day, token):
    day_text = day.isoformat()
    if database.execute("SELECT 1 FROM listed_days WHERE day = ?", (day_text,)).fetchone():
        return

    print(f"{day}: listing public frames", flush=True)
    start = datetime(day.year, day.month, day.day, tzinfo=timezone.utc)
    end = start + ONE_DAY
    query = urlencode({
        # Include midnight even if the API interprets its start bound strictly.
        "start": (start - timedelta(seconds=1)).isoformat(),
        "end": end.isoformat(),
        "public": "true",
        "reduction_level": 91,
        "configuration_type": "EXPOSE",
        "pagination_style": "cursor",
        "limit": 1000 if token else 100,
    })
    next_page = f"/frames/?{query}"
    frames = {}
    while next_page:
        page = get_json(next_page, token)
        for frame in page["results"]:
            observed = datetime.fromisoformat(frame["observation_date"].replace("Z", "+00:00"))
            if start <= observed < end:
                metadata = {field: frame.get(field) for field in FRAME_FIELDS}
                frames[frame["id"]] = json.dumps(metadata)
        next_page = page["next"]

    with database:
        database.executemany(
            "INSERT OR IGNORE INTO frames(id, day, metadata) VALUES (?, ?, ?)",
            ((frame_id, day_text, metadata) for frame_id, metadata in frames.items()),
        )
        database.execute("INSERT OR IGNORE INTO listed_days(day) VALUES (?)", (day_text,))


def print_summary(database, label):
    listed = database.execute("SELECT COUNT(*) FROM listed_days").fetchone()[0]
    saved, total, unavailable = database.execute("""
        SELECT COUNT(headers), COUNT(*), COUNT(header_error) FROM frames
    """).fetchone()
    print(f"{label}: {listed} days listed, {saved}/{total} headers saved", flush=True)
    if unavailable:
        print(f"  {unavailable} headers returned HTTP 404 or 410 and will be retried on the next fetch.",
              flush=True)

In [ ]:
def fetch(args):
    token = None
    if args.authenticated:
        token = os.environ.get("LCO_API_TOKEN", "").strip()
        if not token and sys.stdin.isatty():
            token = getpass.getpass("LCO API token: ").strip()
        if not token:
            raise ValueError("Run --authenticated in an interactive terminal or set LCO_API_TOKEN.")

    attempts = 0
    with closing(open_checkpoint(args.db)) as database:
        try:
            day = args.start
            while day < args.end:
                inventory_day(database, day, token)
                pending = database.execute("""
                    SELECT id FROM frames WHERE day = ? AND headers IS NULL ORDER BY id
                """, (day.isoformat(),)).fetchall()
                remaining = len(pending)
                if args.max_headers_per_day is not None:
                    saved = database.execute(
                        "SELECT COUNT(headers) FROM frames WHERE day = ?", (day.isoformat(),)
                    ).fetchone()[0]
                    remaining = min(remaining, max(0, args.max_headers_per_day - saved))
                    print(f"{day}: {saved} saved, fetching up to {remaining} more", flush=True)
                else:
                    print(f"{day}: {remaining} headers left to fetch", flush=True)
                for (frame_id,) in pending:
                    if remaining == 0:
                        break
                    try:
                        data = get_json(f"/frames/{frame_id}/headers/", token)["data"]
                        if not isinstance(data, dict):
                            raise ValueError(f"Unexpected header response for frame {frame_id}.")
                    except HTTPError as error:
                        if error.code not in (404, 410):
                            raise
                        with database:
                            database.execute("UPDATE frames SET header_error = ? WHERE id = ?",
                                             (error.code, frame_id))
                        print(f"  {frame_id}: header returned HTTP {error.code}", flush=True)
                    else:
                        selected = {field: data.get(field) for field in HEADER_FIELDS}
                        with database:
                            database.execute(
                                "UPDATE frames SET headers = ?, header_error = NULL WHERE id = ?",
                                (json.dumps(selected), frame_id),
                            )
                        remaining -= 1
                    attempts += 1
                    if attempts % 100 == 0:
                        print(f"  {attempts} header requests finished this run", flush=True)
                    if attempts == args.max_headers:
                        print("Pilot limit reached. Re-run the same command to continue.", flush=True)
                        return 0
                day += ONE_DAY
            daily_counts = database.execute("""
                SELECT COUNT(*), COUNT(headers) FROM frames WHERE day >= ? AND day < ? GROUP BY day
            """, (args.start.isoformat(), args.end.isoformat()))
            missing = sum(
                max(0, min(total, args.max_headers_per_day or total) - saved)
                for total, saved in daily_counts
            )
            if missing:
                print(f"Date range is incomplete. {missing} requested headers remain unsaved.", file=sys.stderr)
                return 1
            if args.max_headers_per_day is not None:
                print("Per-day header target reached across the date range.", flush=True)
            else:
                print("Date range complete for the frames listed in this checkpoint.", flush=True)
            return 0
        finally:
            print_summary(database, args.db)

In [ ]:
def export_csv(args):
    if any(args.output.resolve() == checkpoint.resolve() for checkpoint in args.checkpoints):
        raise ValueError("Choose a CSV path different from the checkpoint files.")
    args.output.parent.mkdir(parents=True, exist_ok=True)
    seen = set()
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="", delete=False, dir=args.output.parent,
        ) as output:
            temp_path = Path(output.name)
            writer = csv.DictWriter(output, fieldnames=("id", *FRAME_FIELDS, *HEADER_FIELDS))
            writer.writeheader()
            for checkpoint in args.checkpoints:
                uri = checkpoint.resolve().as_uri() + "?mode=ro"
                with closing(sqlite3.connect(uri, uri=True)) as database:
                    database.execute("BEGIN")
                    rows = database.execute("""
                        SELECT id, metadata, headers FROM frames
                        WHERE headers IS NOT NULL ORDER BY day, id
                    """)
                    for frame_id, metadata, headers in rows:
                        if frame_id in seen:
                            continue
                        writer.writerow({"id": frame_id, **json.loads(metadata), **json.loads(headers)})
                        seen.add(frame_id)
                    print_summary(database, checkpoint)
        temp_path.replace(args.output)
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)
    print(f"Exported {len(seen)} unique frames to {args.output}", flush=True)


def run(function, args):
    """Run fetch/export and report errors the way the command-line version does."""
    try:
        return function(args)
    except KeyboardInterrupt:
        print("\nStopped. Saved headers are in the checkpoint.", file=sys.stderr)
        return 130
    except HTTPError as error:
        if error.code == 429:
            retry_after = error.headers.get("Retry-After")
            detail = f" Retry-After: {retry_after}." if retry_after else ""
            print(f"LCO rate limit reached (HTTP 429).{detail} Resume after it resets.", file=sys.stderr)
        else:
            print(f"LCO request failed: {error}", file=sys.stderr)
        return 1
    except (OSError, ValueError, KeyError, IncompleteRead, sqlite3.Error) as error:
        print(f"Stopped: {error}", file=sys.stderr)
        return 1

## Settings

In [ ]:
# --- Settings: edit these, then run the cells below ---
# Token: asked for once and kept only in this kernel's environment.
if not os.environ.get("LCO_API_TOKEN"):
    os.environ["LCO_API_TOKEN"] = getpass.getpass("LCO token: ")

fetch_args = SimpleNamespace(
    start=date(2025, 2, 1),            # first UTC date
    end=date(2025, 3, 1),              # first excluded UTC date
    db=Path("lco_checkpoint.db"),      # SQLite checkpoint file (re-running resumes from it)
    max_headers=None,                  # e.g. 200 for a pilot run
    max_headers_per_day=None,          # e.g. 300 to cap saved headers per day
    authenticated=True,                # use the token above
)

export_args = SimpleNamespace(
    output=Path("lco_headers.csv"),
    checkpoints=[Path("lco_checkpoint.db")],   # add more .db files here to merge them
)

## Fetch (re-run this cell to resume)

In [ ]:
run(fetch, fetch_args)

## Export to CSV (works on partial runs too)

In [ ]:
run(export_csv, export_args)

## Load the CSV

In [ ]:
import pandas as pd
df = pd.read_csv(export_args.output)
print(df.shape)
df["WCSERR"].value_counts(dropna=False)